# Treasure Hunt Game Notebook — Enhanced

## Original Assignment Context
The theme of this project is a treasure hunt game in which the player needs to find the treasure before the pirate does. The pirate represents an intelligent agent using deep Q-learning to find the optimal path to the treasure

## CS 499 Enhancement Notes
This version adds training visualization, step efficiency tracking and exploration rate comparison on top of the original CS 370 implementation

In [ ]:
from __future__ import print_function
import os, sys, time, datetime, json, random
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import clone_model
from keras.models import Sequential
from keras.layers import Dense, Activation, PReLU
from keras.optimizers import SGD, Adam, RMSprop
import matplotlib.pyplot as plt
from TreasureMaze import TreasureMaze
from GameExperience import GameExperience
%matplotlib inline

## Maze Object Generation
The following 8x8 matrix represents the maze object

In [ ]:
maze = np.array([
    [ 1.,  0.,  1.,  1.,  1.,  1.,  1.,  1.],
    [ 1.,  0.,  1.,  1.,  1.,  0.,  1.,  1.],
    [ 1.,  1.,  1.,  1.,  0.,  1.,  0.,  1.],
    [ 1.,  1.,  1.,  0.,  1.,  1.,  1.,  1.],
    [ 1.,  1.,  0.,  1.,  1.,  1.,  1.,  1.],
    [ 1.,  1.,  1.,  0.,  1.,  0.,  0.,  0.],
    [ 1.,  1.,  1.,  0.,  1.,  1.,  1.,  1.],
    [ 1.,  1.,  1.,  1.,  0.,  1.,  1.,  1.]
])

## Helper Functions and Global Variables
The `show()` function below allows a visual representation of the maze object

In [ ]:
def show(qmaze):
    plt.grid('on')
    nrows, ncols = qmaze.maze.shape
    ax = plt.gca()
    ax.set_xticks(np.arange(0.5, nrows, 1))
    ax.set_yticks(np.arange(0.5, ncols, 1))
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    canvas = np.copy(qmaze.maze)
    for row,col in qmaze.visited:
        canvas[row,col] = 0.6
    pirate_row, pirate_col, _ = qmaze.state
    canvas[pirate_row, pirate_col] = 0.3   # pirate cell
    canvas[nrows-1, ncols-1] = 0.9 # treasure cell
    img = plt.imshow(canvas, interpolation='none', cmap='gray')
    return img

The pirate agent can move in four directions: left, right, up, and down. Exploration vs. exploitation is controlled by epsilon

In [ ]:
LEFT = 0
UP = 1
RIGHT = 2
DOWN = 3

# Exploration factor
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995
patience = 10

# Actions dictionary
actions_dict = {
    LEFT: 'left',
    UP: 'up',
    RIGHT: 'right',
    DOWN: 'down',
}

num_actions = len(actions_dict)

## Playing the Game
`play_game()` simulates a full game using the trained model

**CS 499 Enhancement:** This function now returns a `(result, steps)` tuple instead of just `True`/`False`, so the number of steps taken can be measured as an efficiency metric & not just whether the agent won

In [ ]:
def play_game(model, qmaze, pirate_cell, max_steps=None):
    qmaze.reset(pirate_cell)
    envstate = qmaze.observe()
    steps = 0

    if max_steps is None:
        max_steps = qmaze.maze.size * 4  # safety cutoff

    while steps < max_steps:
        # Only evaluate actions that are legal from the pirate's current position
        valid_actions = qmaze.valid_actions()

        if not valid_actions:
            # enhancement: return step count alongside failure so callers
            # can still see how far agent got before running out of moves
            return False, steps

        state = np.asarray(envstate, dtype=np.float32)

        if state.ndim == 1:
            state = np.expand_dims(state, axis=0)

        q_values = model(state, training=False).numpy()[0]

        # Choose the valid action with the highest predicted Q-value
        valid_q_values = [(a, q_values[a]) for a in valid_actions]
        action = int(max(valid_q_values, key=lambda x: x[1])[0])

        envstate, reward, game_status = qmaze.act(action)
        steps += 1

        if game_status == 'win':
            # enhancement: return step count along with win result
            # so callers can measure efficiency not just win/loss
            return True, steps
        elif game_status == 'lose':
            return False, steps

    return False, steps

## Completion Check
`completion_check()` verifies the trained model can win from every free cell in the maze, not just one starting position.

**CS 499 Enhancement:** Updated to unpack new `(result, steps)` tuple returned by `play_game()`, since this function now needs win/loss result specifically & not the whole tuple

In [ ]:
def completion_check(model, maze_or_qmaze, max_steps=None):
    # Accept either raw numpy maze or TreasureMaze instance
    if isinstance(maze_or_qmaze, TreasureMaze):
        qmaze = maze_or_qmaze
    else:
        qmaze = TreasureMaze(maze_or_qmaze)

    for cell in qmaze.free_cells:
        if not qmaze.valid_actions(cell):
            continue
        # enhancement: play_game() now returns a (result, steps) tuple,
        # unpacked here and only check the win/loss result
        result, steps = play_game(model, qmaze, cell, max_steps=max_steps)
        if not result:
            return False
    return True

## Building the Model
`build_model()` constructs the neural network used by the pirate agent

In [ ]:
def build_model(maze):
    model = Sequential()
    model.add(Dense(maze.size, input_shape=(maze.size,)))
    model.add(PReLU())
    model.add(Dense(maze.size))
    model.add(PReLU())
    model.add(Dense(num_actions))
    model.compile(optimizer='adam', loss='mse')
    return model

## Training Step
`train_step()` performs a single gradient update using the current batch of training data

In [ ]:
loss_fn = tf.keras.losses.MeanSquaredError()
optimizer = tf.keras.optimizers.Adam()

@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        q_values = model(x, training=True)
        loss = loss_fn(y, q_values)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

## Q-Training Algorithm
`qtrain()` implements the deep Q-learning training loop for the pirate agent.

**CS 499 Enhancement:** The function already tracked `win_history`, rolling `win_rate`, `loss` and `best_win_rate` internally, but the data was only printed to the console as text. For the enhancement, `qtrain()` now also builds `win_rate_history` and `loss_history` lists throughout training and returns them at the end, so the data can be plotted in the cell that follows instead of only being visible as a scrolling log. a nwe comparison harness later in the notebook also runs this function multiple times with different exploration settings to compare results side by side

In [ ]:
def qtrain(model, maze, **opt):
    # Exploration factor
    global epsilon
    epsilon = 1.0  # Reset exploration at the start of training
    
    # Number of epochs
    n_epoch = opt.get('n_epoch', 15000)
    
    # Maximum memory to store episodes
    max_memory = opt.get('max_memory', 1000)
    
    # Maximum data size for training
    data_size = opt.get('data_size', 50)
    
    # Frequency of target network updates
    target_update_freq = opt.get('target_update_freq', 50)

    # Frequency for checking full maze completion
    completion_check_freq = opt.get('completion_check_freq', 25)
    
    # Start time
    start_time = datetime.datetime.now()
    
    # Construct environment/game from numpy array
    qmaze = TreasureMaze(maze)
    
    # Target network helps stabilize training
    target_model = clone_model(model)
    target_model.set_weights(model.get_weights())
    
    # Initialize experience replay object
    experience = GameExperience(model, target_model, max_memory=max_memory)

    win_history = []  # History of win/loss games
    hsize = qmaze.maze.size // 2  # History window size
    win_rate = 0.0
    loss = 0.0
    n_episodes = 0

    # CS 499 Enhancement: track win rate and loss per epoch so they can be
    # plotted after training instead of only being printed as text
    win_rate_history = []
    loss_history = []

    # Track the best model weights so late training instability does not overwrite a better policy
    best_win_rate = 0.0
    best_weights = model.get_weights()
    best_epoch = 0
    
    # =============START_HERE================
    for epoch in range(n_epoch):
        
        # rotates through free cells so pirate learns paths from all valid starts
        agent_cell = qmaze.free_cells[epoch % len(qmaze.free_cells)]
        qmaze.reset(agent_cell)
        
        # observes starting environment state
        envstate = qmaze.observe()
        game_status = 'not_over'
        n_episodes = 0
        
        # continues until pirate wins or loses
        while game_status == 'not_over':
            previous_envstate = envstate
            
            # gets valid moves from current position
            valid_actions = qmaze.valid_actions()
            
            # if no valid actions exist, stop episode
            if not valid_actions:
                break
            
            # epsilon-greedy action selection using valid actions
            # exploration chooses random valid move
            # exploitation chooses valid move with highest predicted Q-value
            if np.random.rand() < epsilon:
                action = random.choice(valid_actions)
            else:
                q_values = experience.predict(previous_envstate)

                # restricts exploitation to valid actions so pirate follows legal maze paths
                valid_q_values = [(a, q_values[a]) for a in valid_actions]
                action = int(max(valid_q_values, key=lambda x: x[1])[0])
            
            # applies action and observe new state and reward
            envstate, reward, game_status = qmaze.act(action)
            
            # stores episode in replay memory
            episode = [previous_envstate, action, reward, envstate, game_status]
            experience.remember(episode)
            
            # trains model using replay memory
            inputs, targets = experience.get_data(data_size)
            loss = train_step(inputs, targets)
            n_episodes += 1
        
        # tracks wins and losses
        if game_status == 'win':
            win_history.append(1)
        else:
            win_history.append(0)
        
        # periodically updates target network weights
        if epoch % target_update_freq == 0:
            target_model.set_weights(model.get_weights())
        
        # calculates rolling win rate
        win_rate = sum(win_history[-hsize:]) / hsize if len(win_history) >= hsize else 0.0

        # CS 499 Enhancement: record epoch's win rate and loss for plotting later
        win_rate_history.append(win_rate)
        loss_history.append(float(loss))

        # saves best model weights seen so far
        if win_rate > best_win_rate:
            best_win_rate = win_rate
            best_weights = model.get_weights()
            best_epoch = epoch

        # prints training progress
        dt = datetime.datetime.now() - start_time
        t = format_time(dt.total_seconds())
        print("Epoch: {:03d}/{:d} | Loss: {:.4f} | Episodes: {:d} | Win count: {:d} | Win rate: {:.3f} | Best: {:.3f} | time: {}".format(
            epoch, n_epoch-1, float(loss), n_episodes, sum(win_history), win_rate, best_win_rate, t))

        # gradually reduces exploration while keeping enough randomness
        # to prevent model from getting stuck in poor policies
        epsilon = max(epsilon * epsilon_decay, epsilon_min)
    
        # check full maze completion periodically after rolling win rate is strong
        # this avoids running expensive completion check every epoch
        if win_rate >= 0.95 and epoch % completion_check_freq == 0:
            if completion_check(model, maze):
                best_weights = model.get_weights()
                best_win_rate = win_rate
                best_epoch = epoch
                print(f"Completion check passed at epoch {epoch}")
                break

    # restores best model found during training
    model.set_weights(best_weights)

    total_time = format_time((datetime.datetime.now() - start_time).total_seconds())
    print("Training complete in:", total_time)
    print(f"Best win rate: {best_win_rate:.3f} at epoch {best_epoch}")

    # CS 499 Enhancement: return the tracked history so it can be plotted
    # in a separate cell, rather than only being printed as console text
    return win_rate_history, loss_history


# utility function for readable training time output
def format_time(seconds):
    if seconds < 400:
        s = float(seconds)
        return "%.1f seconds" % (s,)
    elif seconds < 4000:
        m = seconds / 60.0
        return "%.2f minutes" % (m,)
    else:
        h = seconds / 3600.0
        return "%.2f hours" % (h,)

## Plotting Training Performance

**CS 499 Enhancement:** cell plots the win rate and loss history returned by `qtrain()`, giving a visual view of training progress instead of only the printed console log

In [ ]:
model = build_model(maze)
win_rate_history, loss_history = qtrain(model, maze, n_epoch=1000, max_memory=8*maze.size, data_size=32, target_update_freq=50)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(win_rate_history)
ax1.set_title("Win Rate Over Training")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Rolling Win Rate")

ax2.plot(loss_history)
ax2.set_title("Loss Over Training")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")

plt.tight_layout()
plt.show()

## CS 499 Enhancement: Exploration Rate Comparison

The original notebook used a single fixed `epsilon_decay` value with no way to compare how different exploration settings affect training. The enhancement runs training with two different decay rates and plots their win-rate curves against each other

In [ ]:
def run_with_decay(decay_value, n_epoch=1000):
    global epsilon_decay
    original_decay = epsilon_decay
    epsilon_decay = decay_value

    comparison_model = build_model(maze)
    win_rate_hist, loss_hist = qtrain(
        comparison_model, maze,
        n_epoch=n_epoch, max_memory=8*maze.size,
        data_size=32, target_update_freq=50
    )

    epsilon_decay = original_decay  # restore original setting
    return win_rate_hist, loss_hist


# Faster decay: agent shifts to exploitation more quickly
fast_decay_win_rates, fast_decay_loss = run_with_decay(0.90)

# Slower decay: agent keeps exploring for longer before favoring exploitation
slow_decay_win_rates, slow_decay_loss = run_with_decay(0.995)

plt.figure(figsize=(8, 5))
plt.plot(fast_decay_win_rates, label="epsilon_decay = 0.90 (faster)")
plt.plot(slow_decay_win_rates, label="epsilon_decay = 0.995 (slower)")
plt.title("Win Rate Comparison: Exploration Decay Rates")
plt.xlabel("Epoch")
plt.ylabel("Rolling Win Rate")
plt.legend()
plt.show()